In [1]:
import json
import os

def extract_hierarchical_sections(doc_json):
    all_sections = []

    def add_section(sections_stack, new_section):
        """Attach new section at the correct hierarchy level."""
        # If new section is deeper (subheading)
        if sections_stack and new_section["level"] > sections_stack[-1]["level"]:
            sections_stack[-1]["children"].append(new_section)
        else:
            # Pop until we find a parent or root
            while sections_stack and new_section["level"] <= sections_stack[-1]["level"]:
                sections_stack.pop()
            if sections_stack:
                sections_stack[-1]["children"].append(new_section)
            else:
                all_sections.append(new_section)
        sections_stack.append(new_section)

    # Traverse tabs
    for tab in doc_json.get("tabs", []):
        document_tab = tab.get("documentTab")
        if not document_tab:
            continue

        sections_stack = []
        body = document_tab.get("body", {})

        for content in body.get("content", []):
            # --- Paragraphs ---
            if "paragraph" in content:
                paragraph = content["paragraph"]
                style = paragraph.get("paragraphStyle", {}).get("namedStyleType", "NORMAL_TEXT")
                text = "".join(
                    elem.get("textRun", {}).get("content", "")
                    for elem in paragraph.get("elements", [])
                    if "textRun" in elem
                ).strip()

                if not text:
                    continue

                if style.startswith("HEADING_"):
                    level = int(style.split("_")[-1])
                    new_section = {
                        "heading": text,
                        "level": level,
                        "content_parts": [],
                        "children": []
                    }
                    add_section(sections_stack, new_section)
                else:
                    # Add text to current section if exists
                    if sections_stack:
                        sections_stack[-1]["content_parts"].append(text)
                    else:
                        # content before any heading
                        all_sections.append({
                            "heading": None,
                            "level": 0,
                            "content_parts": [text],
                            "children": []
                        })

            # --- Tables ---
            elif "table" in content:
                table = content["table"]
                for row in table.get("tableRows", []):
                    for cell in row.get("tableCells", []):
                        for cell_content in cell.get("content", []):
                            if "paragraph" in cell_content:
                                cell_text = "".join(
                                    elem.get("textRun", {}).get("content", "")
                                    for elem in cell_content["paragraph"].get("elements", [])
                                    if "textRun" in elem
                                ).strip()
                                if cell_text and sections_stack:
                                    sections_stack[-1]["content_parts"].append(cell_text)

    # Clean and transform
    def clean_section(sec):
        """Convert content_parts to single text and recursively clean children."""
        clean_content = " ".join(sec.get("content_parts", [])).replace("\n", " ").strip()
        cleaned = {
            f"heading_{sec['level']}": sec["heading"],
            "content": clean_content
        }
        if sec["children"]:
            cleaned["children"] = [clean_section(c) for c in sec["children"]]
        return cleaned

    cleaned_sections = [clean_section(s) for s in all_sections]
    return cleaned_sections


In [2]:
script_dir = os.getcwd()
data_dir = os.path.join(script_dir, "saved_data")
file_path = os.path.join(data_dir, "raw_text_breast.json")

with open(file_path, "r", encoding="utf-8") as f:
    doc_json = json.load(f)

sections = extract_hierarchical_sections(doc_json)
# get rid of meaningless heading_0 parts
sections = sections[2:]
document = {
    "Category": "Breast",
    "children": sections
}
save_dir = os.path.join(data_dir, "cleaned_text_breast.json")
with open(save_dir, "w", encoding="utf-8") as f:
    json.dump(document, f, indent=4, ensure_ascii=False)